# Спайк 2 — LiveKit turn-detector multilingual (текстовая модель)

Семантический детектор конца реплики по **транскрипту** (от ASR).

- Базовая модель: Qwen2.5-0.5B-Instruct, дистиллирована, ONNX INT8
- Вход: история диалога (до 6 ходов) в формате Qwen chat template,
  у последней реплики пользователя убран закрывающий `<|im_end|>`
- Выход: вероятность того, что дальше идёт `<|im_end|>` = реплика завершена
- 14 языков включая русский, для каждого свой порог в `languages.json`
- HF: https://huggingface.co/livekit/turn-detector

ВАЖНО: мультиязычная модель лежит **не в main**, а в ревизии `v0.4.1-intl`
(файл `onnx/model_q8.onnx` + `languages.json`). В main лежит старая English
SmolLM2-версия — она работает иначе.

## Установка зависимостей

In [1]:
# !pip install onnxruntime transformers huggingface_hub jinja2 numpy

## Загрузка модели (ревизия v0.4.1-intl)

In [2]:
import json
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

REPO = "livekit/turn-detector"
REVISION = "v0.4.1-intl" # мультиязычная модель

tokenizer = AutoTokenizer.from_pretrained(REPO, revision=REVISION)
onnx_path = hf_hub_download(REPO, "onnx/model_q8.onnx", revision=REVISION)
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

# пороги по языкам (подобраны авторами под высокий recall)
languages = json.load(open(hf_hub_download(REPO, "languages.json", revision=REVISION)))
RU_THRESHOLD = languages["ru"]["threshold"]

print("output:", [(o.name, o.shape) for o in session.get_outputs()])
print("ru threshold:", RU_THRESHOLD, "| meta:", languages["ru"])

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


output: [('prob', ['batch_size', 'sequence_length'])]
ru threshold: 0.0032 | meta: {'threshold': 0.0032, 'tpr': 0.9933, 'tnr': 0.8803}


## Функция предсказания

Модель отдаёт вероятность EOU для каждой позиции — берём последнюю.
ONNX уже содержит softmax, дополнительной нормализации не нужно.

In [3]:
def eou_prob(text: str, context: list[dict] | None = None) -> float:
    messages = (context or []) + [{"role": "user", "content": text}]
    s = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    s = s.rsplit("<|im_end|>", 1)[0] # убираем закрывающий тег у последней реплики
    ids = tokenizer(s, return_tensors="np", add_special_tokens=False)["input_ids"].astype(np.int64)
    prob = session.run(None, {"input_ids": ids})[0]   # [1, seq]
    return float(prob[0, -1])

## Тест на русских фразах

In [4]:
tests = [
    ("Здравствуйте, я хотел бы узнать баланс по своей карте", "complete"),
    ("Мне нужно", "incomplete"),
    ("Скажите пожалуйста, какой у меня", "incomplete"),
    ("Я хочу оплатить счёт за электричество", "complete"),
    ("Мой номер телефона восемь девятьсот", "incomplete"),
    ("Спасибо, до свидания", "complete"),
    ("А можно ли", "incomplete"),
    ("Какие у вас есть тарифы?", "complete"),
]

print(f"ru threshold = {RU_THRESHOLD}")
print(f"{'expected':<11}{'score':>8} {'pred':>11}  text")
for text, exp in tests:
    p = eou_prob(text)
    pred = "complete" if p > RU_THRESHOLD else "incomplete"
    mark = "OK" if pred == exp else "XX"
    print(f"{exp:<11}{p:>8.4f} {pred:>11} {mark}  {text}")

ru threshold = 0.0032
expected      score        pred  text
complete     0.4005    complete OK  Здравствуйте, я хотел бы узнать баланс по своей карте
incomplete   0.0070    complete XX  Мне нужно
incomplete   0.0038    complete XX  Скажите пожалуйста, какой у меня
complete     0.1764    complete OK  Я хочу оплатить счёт за электричество
incomplete   0.1221    complete XX  Мой номер телефона восемь девятьсот
complete     0.9444    complete OK  Спасибо, до свидания
incomplete   0.0003  incomplete OK  А можно ли
complete     0.6343    complete OK  Какие у вас есть тарифы?


## Замечание про порог

Дефолтный `ru threshold = 0.0032` подобран авторами под **замену VAD** —
максимальный recall (TPR ~0.99), поэтому почти всё классифицируется как
`complete`.

Score: `"А можно ли"` → ~0.0003, `"Спасибо, до свидания"` → ~0.94.

## Эффект контекста диалога

In [5]:
context = [
    {"role": "assistant", "content": "Назовите номер вашей карты"},
]
phrase = "Пять тысяч двести"
print("без контекста:", round(eou_prob(phrase), 4))
print("с контекстом :", round(eou_prob(phrase, context), 4))

без контекста: 0.2576
с контекстом : 0.7055


## Замер скорости инференса

In [6]:
import time
eou_prob("прогрев")
t0 = time.perf_counter()
N = 50
for _ in range(N):
    eou_prob("Я хочу оплатить счёт за электричество")
print(f"avg latency: {(time.perf_counter()-t0)/N*1000:.1f} ms / вызов")

avg latency: 9.8 ms / вызов


## Массовый тест 

In [7]:
turns = [l.strip() for l in open("../data/ru_turns.txt", encoding="utf-8") if l.strip()]
print("реплик:", len(turns))
turns[:3]

реплик: 4001


['Все же, у здоровяка определенно такого не было.',
 'Конечно. У него грудь, как стена. на ней не было никаких украшений. кроме меток.',
 'Душевно рад вас приветствовать в бостонском отделении.']

In [8]:
import re 
import random
random.seed(0)

def normalize(text: str) -> str:
    """Имитируем ASR"""
    return re.sub(r"[.?!,;:\s]+$", "", text).strip()

N = 2000

samples = []
for t in turns[:N]:
    words = normalize(t).split()
    if len(words) < 5:
        continue
    full = " ".join(words)
    k = random.randint(2, len(words) - 1)
    truncated = " ".join(words[:k])
    samples.append((full, 1))
    samples.append((truncated, 0))

print("всего примеров:", n_samples := len(samples))
print("complete:", n_complete := sum(y for _, y in samples), "| incomplete:", n_incomplete := n_samples - n_complete)
print("\nпример пары:")
print("  + ", samples[0][0])
print("  - ", samples[1][0])

всего примеров: 3430
complete: 1715 | incomplete: 1715

пример пары:
  +  Все же, у здоровяка определенно такого не было
  -  Все же, у здоровяка определенно


In [9]:
import numpy as np

scores = np.array([eou_prob(text) for text, _ in samples])
labels = np.array([y for _, y in samples])

print("avg score complete:", round(scores[labels == 1].mean(), 3))
print("avg score incomplete:", round(scores[labels == 0].mean(), 3))

avg score complete:   0.19
avg score incomplete: 0.079


In [12]:
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

print("ROC-AUC:", round(roc_auc_score(labels, scores), 3))

THRESHOLD = 0.5
preds = (scores > THRESHOLD).astype(int)

cm = confusion_matrix(labels, preds)
print(f"\nconfusion matrix (порог={THRESHOLD}):")
print("                pred_inc  pred_comp")
print("true_incomplete ", cm[0])
print("true_complete   ", cm[1])

print("\n", classification_report(labels, preds, target_names=["incomplete", "complete"]))

ROC-AUC: 0.751

confusion matrix (порог=0.5):
                pred_inc  pred_comp
true_incomplete  [1706    9]
true_complete    [1633   82]

               precision    recall  f1-score   support

  incomplete       0.51      0.99      0.68      1715
    complete       0.90      0.05      0.09      1715

    accuracy                           0.52      3430
   macro avg       0.71      0.52      0.38      3430
weighted avg       0.71      0.52      0.38      3430



In [13]:
print(f"{'thr':>5} {'FP':>4} {'FPR':>6} {'recall':>7} {'precision':>10}")
for thr in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99]:
    preds = (scores > thr).astype(int)
    tp = ((preds == 1) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    tn = ((preds == 0) & (labels == 0)).sum()
    fpr = fp / (fp + tn)# доля incomplete, принятых за complete
    recall = tp / (tp + fn)# сколько complete поймали
    precision = tp / (tp + fp) if tp + fp else 0
    print(f"{thr:>5.2f} {fp:>4} {fpr:>6.3f} {recall:>7.3f} {precision:>10.3f}")

  thr   FP    FPR  recall  precision
 0.30  127  0.074   0.243      0.766
 0.50    9  0.005   0.048      0.901
 0.70    2  0.001   0.003      0.750
 0.80    0  0.000   0.000      0.000
 0.90    0  0.000   0.000      0.000
 0.95    0  0.000   0.000      0.000
 0.99    0  0.000   0.000      0.000


In [14]:
import numpy as np
print("complete   — медиана:", round(np.median(scores[labels==1]), 4),
      "| 90-й перцентиль:", round(np.percentile(scores[labels==1], 90), 4))
print("incomplete — медиана:", round(np.median(scores[labels==0]), 4),
      "| 90-й перцентиль:", round(np.percentile(scores[labels==0], 90), 4))
print("max complete score:", round(scores[labels==1].max(), 4))

complete   — медиана: 0.1594 | 90-й перцентиль: 0.4133
incomplete — медиана: 0.0178 | 90-й перцентиль: 0.2655
max complete score: 0.7596


In [16]:
from sklearn.metrics import roc_auc_score
print("ROC-AUC:", round(roc_auc_score(labels, scores), 3))

ROC-AUC: 0.751


In [17]:
print(f"{'thr':>6} {'FP':>4} {'FPR':>6} {'recall':>7} {'precision':>10}")
for thr in [0.001, 0.003, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2]:
    preds = (scores > thr).astype(int)
    tp = ((preds==1)&(labels==1)).sum(); fp = ((preds==1)&(labels==0)).sum()
    fn = ((preds==0)&(labels==1)).sum(); tn = ((preds==0)&(labels==0)).sum()
    fpr = fp/(fp+tn); recall = tp/(tp+fn); precision = tp/(tp+fp) if tp+fp else 0
    print(f"{thr:>6.3f} {fp:>4} {fpr:>6.3f} {recall:>7.3f} {precision:>10.3f}")

   thr   FP    FPR  recall  precision
 0.001 1377  0.803   0.974      0.548
 0.003 1200  0.700   0.950      0.576
 0.005 1111  0.648   0.938      0.592
 0.010  972  0.567   0.901      0.614
 0.020  837  0.488   0.853      0.636
 0.050  616  0.359   0.753      0.677
 0.100  461  0.269   0.624      0.699
 0.150  350  0.204   0.521      0.719
 0.200  269  0.157   0.414      0.725


### Табличка при пороге 0.2

In [24]:
import pandas as pd 
import numpy as np

pd.set_option("display.max_colwidth", None)

THRESHOLD = 0.2

df = pd.DataFrame({
    "text": [t for t, _ in samples],
    "true": np.where(labels == 1, "complete", "incomplete"),
    "score": scores.round(3),
})
df["pred"] = np.where(df["score"] > THRESHOLD, "complete", "incomplete")
df["correct"] = df["true"] == df["pred"]

print("accuracy:", round(df["correct"].mean(), 3))
df.head(50)

accuracy: 0.629


,text,true,score,pred,correct
0,"Все же, у здоровяка определенно такого не было",complete,0.057,incomplete,False
1,"Все же, у здоровяка определенно",incomplete,0.052,incomplete,True
2,"Конечно. У него грудь, как стена. на ней не было никаких украшений. кроме меток",complete,0.117,incomplete,False
3,"Конечно. У него грудь, как стена. на ней",incomplete,0.157,incomplete,True
4,Душевно рад вас приветствовать в бостонском отделении,complete,0.227,complete,True
5,Душевно рад,incomplete,0.002,incomplete,True
6,И где ж шатается ясной ночкой наш востроглазый братец,complete,0.130,incomplete,False
7,И где ж шатается,incomplete,0.208,complete,False
8,Да по моей просьбе глаза-то и навострил,complete,0.021,incomplete,False
9,Да по моей просьбе глаза-то и,incomplete,0.000,incomplete,True


In [25]:
df[(df.true == "incomplete") & (df.pred == "complete")].sort_values("score", ascending=False).head(20)

,text,true,score,pred,correct
2835,"Может, они в гости пошли?",incomplete,0.719,complete,False
3363,"Нет. идем, сид.",incomplete,0.716,complete,False
1773,"С севера, с белого озера",incomplete,0.646,complete,False
2125,"Присаживайтесь, пожалуйста",incomplete,0.616,complete,False
2495,Но кто поймет меня?,incomplete,0.612,complete,False
2727,"Знаешь, как мне 'знакомство с даром'",incomplete,0.604,complete,False
3069,Лазарь... девчонкам,incomplete,0.557,complete,False
3259,"Диана, Тереза",incomplete,0.520,complete,False
1155,"Как скажешь. но если понадобится помощь, зови. мы подождём",incomplete,0.503,complete,False
105,"Да, это тоже своего рода",incomplete,0.499,complete,False


In [26]:
df[(df.true == "complete") & (df.pred == "incomplete")].sort_values("score").head(20)

,text,true,score,pred,correct
564,А ты рассказала своей семье обо мне,complete,0.0,incomplete,False
1060,"И то, Я тоже, пожалуй, пойду отдохну. надеюсь, завтра мы заключим выгодный для нас обоих договор",complete,0.0,incomplete,False
686,"Настоящее свидание с цветами, ужином и",complete,0.0,incomplete,False
1066,"Трогательная похвала в устах такой женщины, как вы",complete,0.0,incomplete,False
2234,У меня так у деда перед смертью было,complete,0.0,incomplete,False
3044,"За вампирами! ты слушаешь или нет? В общем, эмоционально зависимая Эванджелина, которая донор",complete,0.0,incomplete,False
2932,Она мне не мать! после,complete,0.0,incomplete,False
2106,"Это не тот случай, потому что собаки наши",complete,0.0,incomplete,False
3270,Тем не менее ты все равно прав,complete,0.0,incomplete,False
1986,"У тебя денег не было? что же не сказала, когда я уезжал",complete,0.0,incomplete,False
